**Why Time Series is Unique in the Data Science Workflow**

- Standard Tabular Data: An sssumption that each in a tabular data is independent and indentically distributed.

- Time Series: Shuffling data destroys its meaning entirely

- Today's temperature depends on yesterday's weather.

- Today's retail sales depend on whether it is a Saturday or a Tuesday, and whether a holiday is approaching.

- Causality moves in one direction: The past influences the present, and the present influences the future.


*Standard ML (Order is Independent):*

1. Row[102] ---> Model ---> Prediction

2. Row[4] ---> Model ---> Prediction (Shuffling is fine)


*Time Series ML (Order is Dependent):

1. [t - 2(Jan 1)] ---> [t - 1 (Jan 2)]

**Why Time Series is Critical in Real-World Decision Making**

- Energy & Utilities: Grid operators must forecast hourly electricity demand to prevent blackouts

- Supply Chain & Retail: High-demand item shouldn't run out of stock


**The Anatomy of Time Series**

*The Three Layers of Every Time Series:*

Observed Time Series = Trend + Seasonality + Noise

- Long-Term Trend: The gradual directional movement of time

- Cyclical Seasonality: Repeating Patterns at Fixed Intervals

- Random Noise: Unpredictable Activities


**The Python Toolkit**

- A combination of NumPy's datetime64 & Pandas' Timestam & Datetimeindex

- pd.to_datetime(): Converts arbitrary strings or integers into optimized chronological timestamps

- DatetimeIndex: Index, slice & filter time ranges by dates

- Resampling(resample() vs. asfreq()): Summarizing raw data into regular time buckets

- Time Shifts(shift()): Moves data forward or backward in time

- Rolling Windows(rolling()): A good example is a 7-day moving average

In [1]:
# Using a Raal-World Dataset: NOAA
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

In [3]:
# Set a clean aesthetic styling
plt.style.use('seaborn-v0_8-whitegrid')

In [4]:
import os

# Load the dataset (supports local repo & Google Colab)
data_path = (
    'noaa - 5000.csv' if os.path.exists('noaa - 5000.csv')
    else '/content/drive/MyDrive/Data/noaa - 5000.csv' if os.path.exists('/content/drive/MyDrive/Data/noaa - 5000.csv')
    else 'nooa -1.csv'
)
weather_data = pd.read_csv(data_path)

In [8]:
pd.set_option("display.max_columns", None)

In [9]:
weather_data.head()

,stn,wban,date,year,mo,da,temp,count_temp,dewp,count_dewp,slp,count_slp,stp,count_stp,visib,count_visib,wdsp,count_wdsp,mxpsd,gust,max,flag_max,min,flag_min,prcp,flag_prcp,sndp,fog,rain_drizzle,snow_ice_pellets,hail,thunder,tornado_funnel_cloud
0,10020,99999,2025-07-04,2025,7,4,42.1,4,38.5,4,1004.4,4,3.2,4,999.9,0,7.6,4,9.7,999.9,42.6,NaN,41.2,NaN,0.0,I,999.9,0,0,0,0,0,0
1,10020,99999,2025-04-30,2025,4,30,7.6,4,3.8,4,1002.0,4,0.8,4,999.9,0,18.0,4,23.3,999.9,10.0,NaN,5.7,NaN,0.0,I,999.9,0,0,0,0,0,0
2,10020,99999,2025-04-19,2025,4,19,2.7,4,-2.5,4,1015.5,4,14.3,4,999.9,0,12.8,4,13.6,14.0,4.3,NaN,1.8,NaN,0.0,I,999.9,0,0,0,0,0,0
3,10020,99999,2025-04-20,2025,4,20,4.2,4,-1.0,4,1021.4,4,20.2,4,999.9,0,5.2,4,7.8,999.9,5.9,NaN,3.0,NaN,0.0,I,999.9,0,0,0,0,0,0
4,10020,99999,2025-03-05,2025,3,5,14.4,4,10.0,4,1003.0,4,1.8,4,999.9,0,18.9,4,21.4,999.9,18.7,NaN,9.5,NaN,0.0,I,999.9,0,0,0,0,0,0


In [10]:
weather_data.columns

Index(['stn', 'wban', 'date', 'year', 'mo', 'da', 'temp', 'count_temp', 'dewp',
       'count_dewp', 'slp', 'count_slp', 'stp', 'count_stp', 'visib',
       'count_visib', 'wdsp', 'count_wdsp', 'mxpsd', 'gust', 'max', 'flag_max',
       'min', 'flag_min', 'prcp', 'flag_prcp', 'sndp', 'fog', 'rain_drizzle',
       'snow_ice_pellets', 'hail', 'thunder', 'tornado_funnel_cloud'],
      dtype='object')

In [12]:
# Replace Sentinel codes with NaN
sentinel_cols = ['dewp', 'slp', 'stp', 'visib', 'wdsp', 'mxpsd', 'gust', 'prcp', 'sndp']
for col in sentinel_cols:
  if col in weather_data.columns:
    weather_data[col] = weather_data[col].replace(9999.9, np.nan)

In [13]:
weather_data.head()

,stn,wban,date,year,mo,da,temp,count_temp,dewp,count_dewp,slp,count_slp,stp,count_stp,visib,count_visib,wdsp,count_wdsp,mxpsd,gust,max,flag_max,min,flag_min,prcp,flag_prcp,sndp,fog,rain_drizzle,snow_ice_pellets,hail,thunder,tornado_funnel_cloud
0,10020,99999,2025-07-04,2025,7,4,42.1,4,38.5,4,1004.4,4,3.2,4,999.9,0,7.6,4,9.7,999.9,42.6,NaN,41.2,NaN,0.0,I,999.9,0,0,0,0,0,0
1,10020,99999,2025-04-30,2025,4,30,7.6,4,3.8,4,1002.0,4,0.8,4,999.9,0,18.0,4,23.3,999.9,10.0,NaN,5.7,NaN,0.0,I,999.9,0,0,0,0,0,0
2,10020,99999,2025-04-19,2025,4,19,2.7,4,-2.5,4,1015.5,4,14.3,4,999.9,0,12.8,4,13.6,14.0,4.3,NaN,1.8,NaN,0.0,I,999.9,0,0,0,0,0,0
3,10020,99999,2025-04-20,2025,4,20,4.2,4,-1.0,4,1021.4,4,20.2,4,999.9,0,5.2,4,7.8,999.9,5.9,NaN,3.0,NaN,0.0,I,999.9,0,0,0,0,0,0
4,10020,99999,2025-03-05,2025,3,5,14.4,4,10.0,4,1003.0,4,1.8,4,999.9,0,18.9,4,21.4,999.9,18.7,NaN,9.5,NaN,0.0,I,999.9,0,0,0,0,0,0


In [15]:
weather_data.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 5000 entries, 0 to 4999
Data columns (total 33 columns):
 #   Column                Non-Null Count  Dtype  
---  ------                --------------  -----  
 0   stn                   5000 non-null   int64  
 1   wban                  5000 non-null   int64  
 2   date                  5000 non-null   object 
 3   year                  5000 non-null   int64  
 4   mo                    5000 non-null   int64  
 5   da                    5000 non-null   int64  
 6   temp                  5000 non-null   float64
 7   count_temp            5000 non-null   int64  
 8   dewp                  4765 non-null   float64
 9   count_dewp            5000 non-null   int64  
 10  slp                   3450 non-null   float64
 11  count_slp             5000 non-null   int64  
 12  stp                   5000 non-null   float64
 13  count_stp             5000 non-null   int64  
 14  visib                 5000 non-null   float64
 15  count_visib          

In [16]:
# Convert the date column to DateTimeIndex
weather_data['date'] = pd.to_datetime(weather_data['date'])

In [17]:
# Aggregate daily reading across observation stations
daily_weather = weather_data.groupby('date')[['temp', 'dewp', 'wdsp']].mean()


In [18]:
daily_weather

,temp,dewp,wdsp
date,,,
2025-01-01,50.387500,36.460000,3.031250
2025-01-02,45.115000,34.775000,103.955000
2025-01-03,48.352632,40.531579,162.636842
2025-01-04,43.564286,34.707143,145.264286
2025-01-05,47.028571,43.878947,99.171429
...,...,...,...
2025-08-23,72.822222,60.683333,58.805556
2025-08-24,71.506667,61.686667,4.806667
2025-08-26,59.700000,55.700000,3.900000


In [19]:
print(f"Time Series Range: {daily_weather.index.min().date()} to {daily_weather.index.max().date()}")

Time Series Range: 2025-01-01 to 2025-08-28


In [20]:
print(f"Total Recorded days: {len(daily_weather)}")

Total Recorded days: 239


In [21]:
daily_weather.head()

,temp,dewp,wdsp
date,,,
2025-01-01,50.387500,36.460000,3.031250
2025-01-02,45.115000,34.775000,103.955000
2025-01-03,48.352632,40.531579,162.636842
2025-01-04,43.564286,34.707143,145.264286
2025-01-05,47.028571,43.878947,99.171429


---
## 📌 Session 7 Key Takeaways & Time Series Data-Quality Diagnostics

1. **Chronological Dependence ($t-2 \rightarrow t-1 \rightarrow t$):** Unlike i.i.d. tabular ML where random row shuffling is safe, shuffling time series records destroys autocorrelation and injects look-ahead leakage across train/test boundaries.
2. **Additive Structural Decomposition:** Every real-world time series decomposes into $\text{Observed} = \text{Trend} + \text{Seasonality} + \text{Noise}$, guiding stationarity checks, resampling (`resample()` / `asfreq()`), lag engineering (`shift()`), and moving-window smoothing (`rolling()`).
3. **Multi-Code NOAA Sentinel Audit (`9999.9` vs. `999.9` / `99.99`):** Replacing `9999.9` successfully uncovered `235` missing `dewp` and `1,550` missing `slp` readings across `5,000` station records. Inspecting `daily_weather` reveals that `wdsp` (`999.9` on `2025-08-28`, pulling `2025-01-03` daily mean to `162.6`), `visib`, `gust`, and `sndp` use `999.9` sentinels (and `prcp` uses `99.99`)—critical to sanitize before lag/rolling feature engineering.
4. **Calendar Regularization Readiness:** `daily_weather` contains **`239` recorded dates** across the `240`-day span (`2025-01-01` to `2025-08-28`, missing `2025-08-25`), illustrating why explicit calendar alignment via `.asfreq('D')` / `.resample('D')` is mandatory before applying fixed-window shifts or rolling averages.